In [1]:
# =============================================================================
# Mammography Dataset - Imbalanced Classification
# =============================================================================
# 데이터셋  : mammography (imblearn 내장, IR ≈ 1:42)
# 실행 방법 : 로컬에서 python mammography_classification.py
# 목적      : 불균형 처리 기법 비교 + 분류 모델 성능 평가
# 기법      : No Resampling / SMOTE / ADASYN / SMOTETomek / class_weight
# 모델      : RandomForest, LogisticRegression
# 평가지표  : F1-macro, ROC-AUC, PR-AUC, Recall(1)
# =============================================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.datasets import fetch_datasets
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay
)

# =============================================================================
# 0. 스타일 설정
# =============================================================================
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor':   '#2a2a3e',
    'axes.edgecolor':   '#555577',
    'axes.labelcolor':  '#ccccdd',
    'text.color':       '#ccccdd',
    'xtick.color':      '#aaaacc',
    'ytick.color':      '#aaaacc',
    'grid.color':       '#3a3a5e',
    'grid.alpha':       0.5,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'legend.fontsize':  10,
    'font.family':      'DejaVu Sans',
})

STRATEGY_COLORS = {
    'no_resample':  '#e06c75',
    'smote':        '#61afef',
    'adasyn':       '#98c379',
    'smotetomek':   '#e5c07b',
    'class_weight': '#c678dd',
}
MODEL_COLORS = ['#61afef', '#e5c07b']



In [2]:
# =============================================================================
# 1. 데이터 로드
# =============================================================================
print("=" * 60)
print("1. 데이터 로드")
print("=" * 60)

ds = fetch_datasets()['mammography']
X, y = ds.data, ds.target

# 레이블 변환: -1(정상) → 0,  1(악성) → 1
y = np.where(y == -1, 0, 1)

counts = np.bincount(y)
print(f"Shape          : {X.shape}")
print(f"Class 0 (정상) : {counts[0]:,}  ({counts[0]/len(y)*100:.1f}%)")
print(f"Class 1 (악성) : {counts[1]:,}  ({counts[1]/len(y)*100:.1f}%)")
print(f"IR             : {counts[0]/counts[1]:.1f} : 1")



1. 데이터 로드
Shape          : (11183, 6)
Class 0 (정상) : 10,923  (97.7%)
Class 1 (악성) : 260  (2.3%)
IR             : 42.0 : 1


In [3]:
# =============================================================================
# 2. Train / Test Split  (stratify 필수)
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain : {X_train.shape[0]:,}  class dist → {np.bincount(y_train)}")
print(f"Test  : {X_test.shape[0]:,}  class dist → {np.bincount(y_test)}")

# =============================================================================
# 3. 스케일링
# =============================================================================
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # train 기준 fit
X_test_sc  = scaler.transform(X_test)        # test는 transform만




Train : 8,946  class dist → [8738  208]
Test  : 2,237  class dist → [2185   52]


In [4]:
# =============================================================================
# 4. 리샘플링 전략 정의
# =============================================================================
# ※ class_weight 전략은 리샘플링 없이 모델 파라미터로만 처리
resamplers = {
    'no_resample':  None,
    'smote':        SMOTE(random_state=42),
    'adasyn':       ADASYN(random_state=42),
    'smotetomek':   SMOTETomek(random_state=42),
    'class_weight': None,
}

# =============================================================================
# 5. 모델 정의 헬퍼
# =============================================================================
def get_model(model_name, strategy):
    """strategy가 class_weight이면 class_weight='balanced' 적용"""
    cw = 'balanced' if strategy == 'class_weight' else None
    if model_name == 'RandomForest':
        return RandomForestClassifier(
            n_estimators=100, random_state=42,
            class_weight=cw, n_jobs=-1
        )
    elif model_name == 'LogisticRegression':
        return LogisticRegression(
            max_iter=1000, random_state=42,
            class_weight=cw
        )

MODEL_NAMES = ['RandomForest', 'LogisticRegression']



In [5]:
# =============================================================================
# 6. 학습 & 평가
# =============================================================================
print("\n" + "=" * 60)
print("2. 학습 & 평가")
print("=" * 60)

results       = []    # 성능 지표 누적
conf_mats     = {}    # confusion matrix
roc_store     = {}    # ROC curve 데이터
pr_store      = {}    # PR curve 데이터
resample_dist = {}    # 리샘플링 후 클래스 분포

for strategy, resampler in resamplers.items():

    # ── 리샘플링 (train only) ─────────────────────────────────
    if resampler is not None:
        X_res, y_res = resampler.fit_resample(X_train_sc, y_train)
    else:
        X_res, y_res = X_train_sc.copy(), y_train.copy()

    resample_dist[strategy] = np.bincount(y_res)
    print(f"\n[{strategy}]  train dist → {np.bincount(y_res)}")

    for mname in MODEL_NAMES:
        model = get_model(mname, strategy)
        model.fit(X_res, y_res)

        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:, 1]

        # ── 지표 계산 ──────────────────────────────────────────
        f1      = f1_score(y_test, y_pred, average='macro')
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc  = average_precision_score(y_test, y_proba)
        report  = classification_report(y_test, y_pred, output_dict=True)
        recall1 = report['1']['recall']
        prec1   = report['1']['precision']

        results.append({
            'Strategy':     strategy,
            'Model':        mname,
            'F1-macro':     round(f1, 4),
            'ROC-AUC':      round(roc_auc, 4),
            'PR-AUC':       round(pr_auc, 4),
            'Recall(1)':    round(recall1, 4),
            'Precision(1)': round(prec1, 4),
        })

        key = f"{strategy}|{mname}"
        conf_mats[key] = confusion_matrix(y_test, y_pred)

        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_store[key] = (fpr, tpr, roc_auc)

        prec_c, rec_c, _ = precision_recall_curve(y_test, y_proba)
        pr_store[key] = (rec_c, prec_c, pr_auc)

        print(f"  {mname:22s}  F1={f1:.4f}  ROC={roc_auc:.4f}  "
              f"PR-AUC={pr_auc:.4f}  Recall(1)={recall1:.4f}")

results_df = pd.DataFrame(results)




2. 학습 & 평가

[no_resample]  train dist → [8738  208]
  RandomForest            F1=0.8051  ROC=0.9824  PR-AUC=0.7716  Recall(1)=0.4808
  LogisticRegression      F1=0.7423  ROC=0.9737  PR-AUC=0.6513  Recall(1)=0.3654

[smote]  train dist → [8738 8738]
  RandomForest            F1=0.8477  ROC=0.9878  PR-AUC=0.8199  Recall(1)=0.8462
  LogisticRegression      F1=0.6380  ROC=0.9700  PR-AUC=0.6069  Recall(1)=0.9615

[adasyn]  train dist → [8738 8740]
  RandomForest            F1=0.8284  ROC=0.9692  PR-AUC=0.7323  Recall(1)=0.8269
  LogisticRegression      F1=0.5579  ROC=0.9675  PR-AUC=0.5306  Recall(1)=0.9808

[smotetomek]  train dist → [8715 8715]
  RandomForest            F1=0.8630  ROC=0.9812  PR-AUC=0.8077  Recall(1)=0.8462
  LogisticRegression      F1=0.6387  ROC=0.9702  PR-AUC=0.6078  Recall(1)=0.9615

[class_weight]  train dist → [8738  208]
  RandomForest            F1=0.8051  ROC=0.9662  PR-AUC=0.7727  Recall(1)=0.4808
  LogisticRegression      F1=0.6335  ROC=0.9670  PR-AUC=0.6133  R

In [6]:
# =============================================================================
# 7. 시각화
# =============================================================================
print("\n" + "=" * 60)
print("3. 시각화 생성")
print("=" * 60)

# ── Fig 1: 클래스 분포 (원본 + 리샘플링 후) ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Mammography — Class Distribution',
             fontsize=15, fontweight='bold', color='#abb2bf')

# 원본
ax = axes[0]
bars = ax.bar(['Normal (0)', 'Malignant (1)'], counts,
              color=['#61afef', '#e06c75'], width=0.45,
              edgecolor='white', linewidth=0.8)
ax.set_title('Original Dataset', fontweight='bold')
ax.set_ylabel('Count')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 150,
            f'{cnt:,}\n({cnt/len(y)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, color='white')
ax.set_ylim(0, counts[0] * 1.3)

# 리샘플링 후 비교
ax2 = axes[1]
strats_with_resample = [s for s, r in resamplers.items() if r is not None]
x = np.arange(len(strats_with_resample))
w = 0.35
n_vals = [resample_dist[s][0] for s in strats_with_resample]
m_vals = [resample_dist[s][1] for s in strats_with_resample]
ax2.bar(x - w/2, n_vals, w, label='Normal (0)',    color='#61afef', alpha=0.85)
ax2.bar(x + w/2, m_vals, w, label='Malignant (1)', color='#e06c75', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(strats_with_resample, rotation=10)
ax2.set_title('After Resampling (Train Set)', fontweight='bold')
ax2.set_ylabel('Count')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("  fig1_class_distribution.png")

# ── Fig 2: 성능 히트맵 ──────────────────────────────────────────
metrics_cols = ['F1-macro', 'ROC-AUC', 'PR-AUC', 'Recall(1)', 'Precision(1)']
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Performance Heatmap by Strategy & Model',
             fontsize=15, fontweight='bold', color='#abb2bf')

for ax, mname in zip(axes, MODEL_NAMES):
    sub = (results_df[results_df['Model'] == mname]
           [['Strategy'] + metrics_cols]
           .set_index('Strategy')
           .astype(float))
    sns.heatmap(sub, ax=ax, annot=True, fmt='.4f',
                cmap='YlOrRd', linewidths=0.5, linecolor='#1e1e2e',
                cbar_kws={'shrink': 0.8}, annot_kws={'size': 11})
    ax.set_title(mname, fontweight='bold', fontsize=13)
    ax.tick_params(axis='x', rotation=30)
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('fig2_performance_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("  fig2_performance_heatmap.png")

# ── Fig 3: ROC & PR Curve (RandomForest 기준) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ROC & PR Curves — RandomForest',
             fontsize=15, fontweight='bold', color='#abb2bf')

ax_roc, ax_pr = axes

for strategy, color in STRATEGY_COLORS.items():
    key = f"{strategy}|RandomForest"
    if key not in roc_store:
        continue
    fpr, tpr, auc_v = roc_store[key]
    ax_roc.plot(fpr, tpr, color=color, lw=2,
                label=f'{strategy} (AUC={auc_v:.3f})')

    rec_c, prec_c, ap_v = pr_store[key]
    ax_pr.plot(rec_c, prec_c, color=color, lw=2,
               label=f'{strategy} (AP={ap_v:.3f})')

ax_roc.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.4, label='Random')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curve', fontweight='bold')
ax_roc.legend(loc='lower right', fontsize=9)
ax_roc.grid(True, alpha=0.3)

baseline = counts[1] / len(y)
ax_pr.axhline(baseline, color='white', linestyle='--', lw=1,
              alpha=0.4, label=f'Baseline ({baseline:.3f})')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.set_title('Precision-Recall Curve', fontweight='bold')
ax_pr.legend(loc='upper right', fontsize=9)
ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("  fig3_roc_pr_curves.png")

# ── Fig 4: Confusion Matrix 5전략 × 2모델 ───────────────────────
n_strat = len(resamplers)
for mname in MODEL_NAMES:
    fig, axes = plt.subplots(1, n_strat, figsize=(4.5 * n_strat, 4.5))
    fig.suptitle(f'Confusion Matrices — {mname}',
                 fontsize=14, fontweight='bold', color='#abb2bf')
    for ax, (strategy, _) in zip(axes, resamplers.items()):
        key = f"{strategy}|{mname}"
        cm = conf_mats[key]
        disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Malignant'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(strategy, fontweight='bold', fontsize=11)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    plt.tight_layout()
    fname = f'fig4_confusion_{mname.lower()}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  {fname}")

# ── Fig 5: Recall(1) & PR-AUC 전략별 바 차트 ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Recall(Malignant) & PR-AUC by Strategy',
             fontsize=14, fontweight='bold', color='#abb2bf')

for ax, metric in zip(axes, ['Recall(1)', 'PR-AUC']):
    strategies_list = results_df['Strategy'].unique()
    x = np.arange(len(strategies_list))
    w = 0.35
    for i, (mname, color) in enumerate(zip(MODEL_NAMES, MODEL_COLORS)):
        sub  = results_df[results_df['Model'] == mname].set_index('Strategy')
        vals = [sub.loc[s, metric] for s in strategies_list]
        offset = (i - 0.5) * w
        bars = ax.bar(x + offset, vals, w, label=mname,
                      color=color, alpha=0.88,
                      edgecolor='white', linewidth=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.008,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=8, color='white')
    ax.set_xticks(x)
    ax.set_xticklabels(strategies_list, rotation=12)
    ax.set_ylabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.18)
    ax.axhline(0.8, color='#e06c75', linestyle='--', alpha=0.5, linewidth=1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig5_recall_prauc_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("  fig5_recall_prauc_comparison.png")

# =============================================================================
# 8. 최종 결과 요약
# =============================================================================
print("\n" + "=" * 60)
print("4. 최종 성능 비교표")
print("=" * 60)
print(results_df.to_string(index=False))

best_pr  = results_df.loc[results_df['PR-AUC'].idxmax()]
best_rec = results_df.loc[results_df['Recall(1)'].idxmax()]
best_f1  = results_df.loc[results_df['F1-macro'].idxmax()]

print(f"\n★ PR-AUC   최고 : [{best_pr['Strategy']}]  {best_pr['Model']}"
      f"  PR-AUC={best_pr['PR-AUC']}")
print(f"★ Recall   최고 : [{best_rec['Strategy']}]  {best_rec['Model']}"
      f"  Recall(1)={best_rec['Recall(1)']}")
print(f"★ F1-macro 최고 : [{best_f1['Strategy']}]  {best_f1['Model']}"
      f"  F1={best_f1['F1-macro']}")
print("\n완료. 그래프 파일 6종이 현재 디렉토리에 저장되었습니다.")


3. 시각화 생성
  fig1_class_distribution.png
  fig2_performance_heatmap.png
  fig3_roc_pr_curves.png
  fig4_confusion_randomforest.png
  fig4_confusion_logisticregression.png
  fig5_recall_prauc_comparison.png

4. 최종 성능 비교표
    Strategy              Model  F1-macro  ROC-AUC  PR-AUC  Recall(1)  Precision(1)
 no_resample       RandomForest    0.8051   0.9824  0.7716     0.4808        0.8621
 no_resample LogisticRegression    0.7423   0.9737  0.6513     0.3654        0.7600
       smote       RandomForest    0.8477   0.9878  0.8199     0.8462        0.6027
       smote LogisticRegression    0.6380   0.9700  0.6069     0.9615        0.1961
      adasyn       RandomForest    0.8284   0.9692  0.7323     0.8269        0.5584
      adasyn LogisticRegression    0.5579   0.9675  0.5306     0.9808        0.1183
  smotetomek       RandomForest    0.8630   0.9812  0.8077     0.8462        0.6471
  smotetomek LogisticRegression    0.6387   0.9702  0.6078     0.9615        0.1969
class_weight       Rando